In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import collections
import itertools
import functools
import os
import pathlib
import shutil
import datetime

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.decomposition
import plotly.express as px
import scipy
import statsmodels.stats.multitest
import decoupler

import common_data
import common_plots
import factor_interpretation as fi

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"
# mpl.rcParams['figure.constrained_layout.use'] = True

# Pathogen pseudobulk principal component analysis

## Goals:

1. For each cell type
   1. Select top $N$ variable features (per DESeq2, maybe not needed)
   2. Run PCA
   3. Save & plot elbow plot
   4. Save loadings
   5. Plot interactive PC with plotly
   6. Correlate to $P$ PCs with numerical features + run ANOVA for pathogen categories
   7. Run ANOVA for multiple levels of pathogens

In [ ]:
ROOT = common_data.DATA
BASE = ROOT / '05_pseudobulk/01_all_pseudobulk'

In [6]:
TOP_N_GENES = 2000
TOP_N_PCS = 10

In [7]:
msigdb = decoupler.get_resource('MSigDB', organism='human')

In [8]:
hallmark = msigdb[msigdb.collection.eq('hallmark')]

In [9]:
hallmark = hallmark[~hallmark.duplicated(['geneset', 'genesymbol'])]

In [10]:
sc_meta = pd.read_csv(
    common_data.SC_META,
    index_col=0
)

/tmp/ipykernel_1911610/2406962787.py:1: DtypeWarning: Columns (16,17,25,31,32,35,37,40,41,52,54,56,57,58,66) have mixed types. Specify dtype option on import or set low_memory=False.
  sc_meta = pd.read_csv(


In [11]:
pseudobulk_n_cells = sc_meta.groupby(['individual', 'Level_6']).size().reset_index(name='n_cells')

In [12]:
pseudobulk_n_cells = pseudobulk_n_cells.loc[pseudobulk_n_cells.n_cells.ge(50)].copy()

In [13]:
pseudobulk_n_cells.rename(columns={'individual': 'sample', 'Level_6': 'cell_type'}, inplace=True)

In [14]:
sample_meta = sc_meta.groupby('individual').head(1)[
    ['days_on_ventilator', 'sequencing_depth', 'sequencing_saturation',
    'frac_reads_in_cells', 'viability', 'individual']
].set_index('individual')

In [17]:
categorical_covariates = common_data.get_sc_categorical_covariates()

In [18]:
numerical_covariates = common_data.get_sc_numerical_covariates()

In [19]:
na_values = {col: 'NA' for col in categorical_covariates.columns}
na_values.update(common_data.na_values)
na_values['Pathogen_groups'] = ['Healthy', 'NPC', 'discard']

In [20]:
categorical_covariates['Pathogen_groups'] = categorical_covariates[
    'Pathogen_groups'
].cat.rename_categories({
    'Pseudomonas aeruginosa': 'Pseudomonas',
    'Pseudomonas aeruginosa; SARS-CoV-2': 'Pseudomonas; SARS-CoV-2'
})

In [21]:
healthy_samples = categorical_covariates.index[categorical_covariates.Pathogen_groups.eq('Healthy')]

In [ ]:
class CellTypeInfo:
    def __init__(self, path, additional_meta):
        self.path = path
        self.comparisons = []
        self.meta = pd.read_csv(path / 'meta.csv', index_col=0)
        self.meta = self.meta.merge(additional_meta, left_on='sample', right_index=True)
        for c in self.meta.columns:
            if pd.api.types.is_categorical_dtype(self.meta[c]):
                self.meta[c] = self.meta[c].cat.remove_unused_categories()
        self.meta['is_healthy'] = self.meta['sample'].isin(healthy_samples)
        self.name = self.meta.cell_type.values[0]
        self.vst = pd.read_table(path / 'transformed.tsv', delim_whitespace=True).T
        # subset
        self.meta = self.meta.loc[self.meta['sample'].isin(additional_meta.index)]
        self.vst = self.vst.loc[self.meta['sample']]
        self.run_pca()

    def select_top_genes(self):
        top_genes = self.vst.var(axis=0).sort_values().index[-TOP_N_GENES:]
        return self.vst.loc[:, top_genes].copy()

    def run_pca(self):
        self.top_vst = self.select_top_genes()
        self.pca = sklearn.decomposition.PCA()
        self.pcs = self.pca.fit_transform(self.top_vst)

    def plot_pca_elbow(self):
        to_plot = min(50, self.pca.n_components_)
        fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
        ax.bar(range(to_plot), self.pca.explained_variance_ratio_[:to_plot] * 100)
        ax.set_xlabel('PC', size=14)
        ax.set_ylabel('% variance explained\n(top 2000 genes)', size=14)
        ax.set_title(self.name, size=16)
        return fig

    def get_pc_weights(self):
        n_pcs = min(self.pcs.shape[1], TOP_N_PCS)
        return pd.DataFrame(
            self.pca.components_[:n_pcs].T,
            index=self.top_vst.columns,
            columns='PC' + (pd.Series(range(n_pcs)) + 1).astype(str)
        ).sort_values('PC1')

    def get_pc_df(self, additional_meta):
        n_pcs = min(self.pcs.shape[1], TOP_N_PCS)
        pc_df = self.meta.copy()
        pc_df.loc[:, 'PC' + (pd.Series(range(n_pcs)) + 1).astype(str)] = self.pcs[:, :n_pcs]
        pc_df['size'] = pc_df.days_on_ventilator.copy()
        pc_df.fillna(0, inplace=True)
        pc_df.loc[pc_df['size'].lt(0), 'size'] = 0
        pc_df['size'] = (pc_df['size'] + 0.5) ** 0.5
        return pc_df

In [28]:
all_covariates = categorical_covariates.merge(numerical_covariates, left_index=True, right_index=True)

In [29]:
%%time
data = []
for cell_type_path in sorted(BASE.iterdir()):
    if not (cell_type_path / 'transformed.tsv').exists():
        continue
    info = CellTypeInfo(cell_type_path, all_covariates)
    data.append(info)

CPU times: user 14.9 s, sys: 30.1 s, total: 45 s
Wall time: 7.77 s


In [30]:
[(i.name, i.get_pc_df(all_covariates).shape) for i in data]

[('AT1 and AT2', (8, 78)),
 ('B cells', (60, 80)),
 ('CD4 T cells', (265, 80)),
 ('CD8 T cells', (277, 80)),
 ('Ciliated cells', (70, 80)),
 ('Classical monocytes-1 CCR2', (89, 80)),
 ('Classical monocytes-2 IL1B', (236, 80)),
 ('DC1', (22, 80)),
 ('DC2', (167, 80)),
 ('Hematopoietic stem cells', (6, 76)),
 ('MRC1+C1QA+', (294, 80)),
 ('MRC1+C1QA-', (296, 80)),
 ('Mast cells', (19, 80)),
 ('Migratory DC', (12, 80)),
 ('NUPR1+ Macs', (265, 80)),
 ('Non-classical monocytes', (31, 80)),
 ('Perivascular macrophages', (126, 80)),
 ('Plasma cells', (59, 80)),
 ('Proliferating CD4 T cells', (85, 80)),
 ('Proliferating CD8 T cells', (82, 80)),
 ('Proliferating NUPR1+ Macs', (115, 80)),
 ('Proliferating gdT cells', (11, 80)),
 ('Proliferating plasma cells', (69, 80)),
 ('Secretory cells', (47, 80)),
 ('Tregs', (123, 80)),
 ('gdT cells', (178, 80)),
 ('pDC', (25, 80))]

In [40]:
def plot_pc(cell_type, pc_df):
    pc1_label = f'PC1 {cell_type.pca.explained_variance_ratio_[0] * 100:.1f}%'
    pc2_label = f'PC2 {cell_type.pca.explained_variance_ratio_[1] * 100:.1f}%'
    return px.scatter(
        pc_df,
        x='PC1',
        y='PC2',
        color='Pathogen_groups',
        size='size',
        hover_data=dict(
            sample=True,
            days_on_ventilator=True,
            size=False,
            Sex=True,
            VAP_onset_within_7d=True,
            VAP_is_cured_d7=True,
            PC1=False,
            PC2=False,
        ),
        color_discrete_map=common_plots.PATHOGEN_PALETTE,
        labels=dict(
            PC1=pc1_label,
            PC2=pc2_label
        ),
        category_orders=dict(
            Pathogen_groups=common_plots.PATHOGEN_ORDER
        ),
        title=cell_type.name
    )

In [ ]:
def plot_pc_cat_boxplot(name, pc_df, pc, cat, na_values):
    fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
    stats_results = []

    cat_na_values = na_values.get(cat)
    if cat_na_values is None:
        cat_na_values = []
    if type(cat_na_values) is str:
        cat_na_values = [cat_na_values]

    categories = list(pc_df[cat].cat.categories)
    new_categories = []
    new_na = []
    for c in categories:
        if c in cat_na_values:
            new_na.append(c)
        else:
            new_categories.append(c)
    new_categories.extend(new_na)
    pc_df[cat] = pc_df[cat].cat.reorder_categories(new_categories)

    for d1, d2 in itertools.combinations(pc_df[cat].unique(), 2):
        if d1 in cat_na_values or d2 in cat_na_values:
            continue
        days1 = pc_df[pc][pc_df[cat].eq(d1)].dropna()
        days2 = pc_df[pc][pc_df[cat].eq(d2)].dropna()
        if days1.size == 0 or days2.size == 0:
            continue
        pval = scipy.stats.mannwhitneyu(days1, days2).pvalue
        stats_results.append([d1, d2, days1.size, days2.size, pval])

    stats_results = pd.DataFrame(
        stats_results,
        columns=["group1", "group2", "group1_size", "group2_size", "pval"]
    )
    stats_results['padj'] = statsmodels.stats.multitest.fdrcorrection(stats_results.pval)[1]
    stats_results['PC'] = pc
    stats_results['covariate'] = cat
    stats_results_sign = stats_results.loc[stats_results.padj.lt(0.05)]

    sns.boxplot(data=pc_df, x=cat, y=pc, ax=ax, color='lightgray')
    sns.swarmplot(data=pc_df, x=cat, y=pc, ax=ax, size=2, hue='is_healthy', palette={
        True: 'red',
        False: 'black'
    }, legend=False)

    start_height = pc_df[pc].max()
    incrementer = 15 # px
    labels = [x.get_text() for x in ax.get_xticklabels()]
    q = ax.transData.inverted().transform([[0, 0], [0, incrementer]])
    y_offset = q[1][1] - q[0][1]
    gap = y_offset / 2
    y = start_height
    for _, r in stats_results_sign.iterrows():
        p = f'{r.padj:.3f}'

        # statistical annotation
        try:
            x1, x2 = labels.index(str(r.group1)), labels.index(str(r.group2))
        except:
            if type(r.group1) == float:
                group1 = int(r.group1)
            if type(r.group1) == float:
                group2 = int(r.group1)
            x1, x2 = labels.index(str(group1)), labels.index(str(group2))
        col = 'k'
        h = gap
        y += gap

        bracket = ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=0.5, c=col)
        txt = ax.text((x1+x2)*.5, y+.95*h, p, ha='center', va='bottom', color=col, size=6)
        y += y_offset

    ax.set_title(f'{cell_type.name}: {pc} vs {cat}', size=14)

    max_l = max(*[len(val) for val in pc_df[cat].cat.categories])
    if max_l > 15:
        trans = mpl.transforms.Affine2D().translate(6, 0)
        for t in ax.get_xticklabels():
            t.set_rotation(30)
            t.set_horizontalalignment("right")
            t.set_transform(t.get_transform() + trans)

    return fig, stats_results

In [ ]:
# TODO: run more iterations in parallel in jobs
def test_pc_gsea(pc_loadings, gene_network):
    result = []
    for pc in pc_loadings.columns:
        enriched = decoupler.get_gsea_df(
            df=pc_loadings,
            stat=pc,
            net=hallmark,
            source='geneset',
            target='genesymbol',
            times=10_000,
            seed=1066
        )
        enriched['pc'] = pc
        result.append(enriched)
    result = pd.concat(result)
    idx = result['NOM p-value'].eq(0)
    min_p_val = result['NOM p-value'][~idx].min()
    result.loc[idx, 'NOM p-value'] = min_p_val * 0.01
    print(f'P-values exactly 0: {idx.sum()}')

    result['padj'] = statsmodels.stats.multitest.fdrcorrection(result['NOM p-value'])[1]
    result['-log10(padj)'] = -np.log10(result.padj)
    result['-log10(padj)'] *= np.sign(result.NES)
    result = result.pivot_table(
        index='pc',
        columns='Term',
        values='-log10(padj)'
    )
    return result.T

In [ ]:
%%time
all_pairwise = []
for cell_type in data:
    print(cell_type.name)
    os.makedirs(cell_type.path / 'pca', exist_ok=True)
    fig = cell_type.plot_pca_elbow()
    fig.savefig(cell_type.path / 'pca/pca_elbow.pdf')
    plt.close()

    weights = cell_type.get_pc_weights().round(5)
    weights.to_csv(cell_type.path / 'pca/pc_loadings.csv')
    enr = test_pc_gsea(weights, hallmark)
    enr_plot = common_plots.plot_pc_gsea(enr)
    enr_plot.fig.savefig(cell_type.path / 'pca/pc_loadings_gsea.pdf')
    plt.close()

    pc_df = cell_type.get_pc_df(sample_meta)
    fig = plot_pc(cell_type, pc_df)
    fig.write_html(cell_type.path / 'pca/pc_plot.html')

    pc_names = pc_df.columns[pc_df.columns.str.startswith('PC')]
    pc_df = pc_df.set_index('sample')

    cat_covs = categorical_covariates.loc[pc_df.index].copy()
    for col in cat_covs.columns:
        cat_covs[col] = cat_covs[col].cat.remove_unused_categories()

    pc_cat_assoc = fi.test_association(
        pc_df.loc[:, pc_names],
        cat_covs,
        na_values,
        alternative_covariates=common_data.alternative_covariates
    )
    cg = common_plots.plot_pc_cat_assoc(pc_cat_assoc, cell_type.name)
    cg.fig.savefig(cell_type.path / 'pca/pc_cat_assoc.pdf')
    plt.close()

    pc_corrs, pc_corr_pvals = fi.test_correlation(
        pc_df.loc[:, pc_names],
        numerical_covariates.loc[pc_df.index],
        na_values=dict(
            sequencing_depth=-1,
            sequencing_saturation=-1,
            frac_reads_in_cells=-1,
            viability=-1,
        )
    )
    cg = common_plots.plot_pc_corrs(pc_corrs, pc_corr_pvals, cell_type.name)
    cg.fig.savefig(cell_type.path / 'pca/pc_corrs.pdf')
    plt.close()

    stat_results = []
    for pc, cat in pc_cat_assoc.where(pc_cat_assoc < 0.05).stack().index.values:
        os.makedirs(cell_type.path / 'pca/pc_cat_assoc', exist_ok=True)

        fig, stats = plot_pc_cat_boxplot(cell_type.name, pc_df, pc, cat, na_values)
        fig.savefig(cell_type.path / f'pca/pc_cat_assoc/{pc}_{cat}.pdf')
        plt.close()
        stat_results.append(stats)
    if len(stat_results) > 0:
        stat_results = pd.concat(stat_results)
        stat_results.insert(0, 'cell_type', cell_type.name)
        all_pairwise.append(stat_results)
all_pairwise = pd.concat(all_pairwise)
all_pairwise.to_csv(BASE / '_pc_pairwise_covariate_tests.csv')

### Copy files to simpler folder structure for publishing

In [42]:
HTML = """
<!DOCTYPE html>
<html>
<head>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <meta http-equiv="content-type" content="text/html; charset=utf8"/>
  <style type="text/css">
    html {
      margin: 0; padding: 0;
      font-size: 20px; font-family: Helvetica, Verdana, sans-serif;
    }
    .header {margin-bottom: 10px;}
    .header h2, .header h3 {font-weight: normal; text-align: center; margin: 0 0 6px 0;}
    body {margin: 0; padding: 50px 100px;}
    a {color: #1385cb}
    a:visited {color: #0e74bc}
    table {border-collapse: collapse; border-top: 1px solid #ccc; border-left: 1px solid #ccc;}
    table td, table th {padding: 2px 5px; border-bottom: 1px solid #ccc; border-right: 1px solid #ccc;}
    table td {font-size: 14px;}
  </style>
</head>
<body>
<div class="header">
    <h2>PCA analysis</h2>
    Generated on %s
</div>
<table>
%s
</table>
</body>
</html>
"""

HTML_CELL_TYPE = """
<tr>
    <th>%s</th>
    <td><a href="%s" target="_blank">PC plot</a></td>
    <td><a href="%s" target="_blank">PC elbow plot</a></td>
    <td><a href="%s" target="_blank">PC loadings table</a></td>
    <td><a href="%s" target="_blank">PC loadings GSEA</a></td>
    <td><a href="%s" target="_blank">PC categorical associations</a></td>
    <td><a href="%s" target="_blank">PC categorical boxplots</a></td>
    <td><a href="%s" target="_blank">PC numerical associations</a></td>
</tr>
"""

In [43]:
def sanitize_name(name):
    return name.replace(' ', '_').replace('*', '').replace(';', '_and').replace('/', '_')

In [ ]:
TARGET = common_data.DATA / '05_pseudobulk/01_pca'
table = ''
for cell_type in data:
    dest = TARGET / cell_type.name
    os.makedirs(dest, exist_ok=True)
    shutil.copy2(cell_type.path / 'pca/pca_elbow.pdf', dest)
    shutil.copy2(cell_type.path / 'pca/pc_loadings.csv', dest / f'{sanitize_name(cell_type.name)}_pc_loadings.csv')
    shutil.copy2(cell_type.path / 'pca/pc_loadings_gsea.pdf', dest)
    shutil.copy2(cell_type.path / 'pca/pc_plot.html', dest)
    shutil.copy2(cell_type.path / 'pca/pc_cat_assoc.pdf', dest)
    shutil.copy2(cell_type.path / 'pca/pc_corrs.pdf', dest)
    if (cell_type.path / 'pca/pc_cat_assoc').exists():
        shutil.copytree(
            cell_type.path / 'pca/pc_cat_assoc',
            dest / 'pc_cat_assoc',
            dirs_exist_ok=True
        )
    cell_html = HTML_CELL_TYPE % (
        cell_type.name,
        cell_type.name + '/pc_plot.html',
        cell_type.name + '/pca_elbow.pdf',
        cell_type.name + f'/{sanitize_name(cell_type.name)}_pc_loadings.csv',
        cell_type.name + '/pc_loadings_gsea.pdf',
        cell_type.name + '/pc_cat_assoc.pdf',
        cell_type.name + '/pc_cat_assoc/',
        cell_type.name + '/pc_corrs.pdf',
    )
    table += cell_html
html = HTML % (
    datetime.datetime.now().strftime('%d %b %Y, %I:%M%p'),
    table
)
with open(TARGET / 'index.html', 'w') as out:
    out.write(html)